In [1]:
from sqlalchemy import create_engine, text
from concurrent.futures import ThreadPoolExecutor, as_completed
from DATA.stock_invest_function import *
import logging
import time

In [2]:
# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# =============================================================================
# 설정 부분 - 여기를 수정하세요
# =============================================================================
SERVICE_KEY = '2o6NG3ixxDgGQ9S4dWUgsMac9WlxfX46%2BJvFRsAlsXQ6xVi6CZewvNJvbHd4S7exkWwt3YWoKSdwvUNb46kSTQ%3D%3D'
# HS_CODES = ['854231', '848690']  # 실제 HS 코드 입력
START_YEAR = 2008
END_YEAR = 2026
REGION_NAME = '전국'

# DB 설정 (필요시)
RETRIES = 3             # 재시도 횟수
DB_INFO = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

hs_data = fetch_table_data(DB_INFO, "target_hs_code")
HS_CODES = hs_data['hs_code'].unique().tolist()

✅ 'target_hs_code' 테이블에서 808건의 데이터를 가져왔습니다.


In [ ]:
# 성능 설정
MAX_WORKERS = 15        # 동시 요청 수
REQUEST_DELAY = 0.05    # API 요청 간격 (초)
CHUNK_SIZE = 5000       # DB 업로드 청크 크기
TIMEOUT = 30            # API 요청 타임아웃

# =============================================================================
# 유틸리티 함수들
# =============================================================================

def get_period_list(start, end, last_month):
    """기간 리스트 생성"""
    cut_num = last_month - 12
    end = end + 1

    period_list = []
    for y in range(start, end):
        for m in range(1, 13):
            m_str = f"{m:02d}"
            period_list.append(f"{y}{m_str}")

    return period_list[:cut_num]

def get_country_export_by_item_optimized(session, service_key, start, end, hs_code):
    """단일 HS 코드에 대한 데이터 수집 (최적화)"""
    url = (f'https://apis.data.go.kr/1220000/Itemtrade/getItemtradeList'
           f'?serviceKey={service_key}&strtYymm={start}&endYymm={end}&hsSgn={hs_code}')

    for attempt in range(RETRIES):
        try:
            response = session.get(url, timeout=TIMEOUT)
            response.raise_for_status()

            # XML to JSON 변환
            json_dict = json.loads(json.dumps(xmltodict.parse(response.text), indent=4))
            items = json_dict.get('response', {}).get('body', {}).get('items')

            if items is None or items.get('item') is None:
                logger.warning(f"No data for HS {hs_code} from {start} to {end}")
                return pd.DataFrame()

            # DataFrame 생성
            item_data = items['item']
            if isinstance(item_data, dict):  # 단일 레코드인 경우
                item_data = [item_data]

            df = pd.DataFrame(item_data)
            df['root_hs_code'] = hs_code

            # API 요청 간격 조절
            if REQUEST_DELAY > 0:
                time.sleep(REQUEST_DELAY)

            return df

        except requests.exceptions.RequestException as e:
            logger.warning(f"Attempt {attempt + 1} failed for HS {hs_code}: {e}")
            if attempt < RETRIES - 1:
                time.sleep(2 ** attempt)  # 지수적 백오프
            else:
                logger.error(f"All attempts failed for HS {hs_code}")
                return pd.DataFrame()
        except Exception as e:
            logger.error(f"Unexpected error for HS {hs_code}: {e}")
            return pd.DataFrame()

def process_and_aggregate(df, region_name):
    """데이터 전처리 및 집계"""
    # 총계 데이터 제거
    df = df[df['year'] != '총계'].copy()

    # 날짜 처리
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        df['new_date'] = pd.to_datetime(df['year'].str.replace('.', '-'), errors='coerce') + MonthEnd(0)

    # 유효하지 않은 날짜 제거
    df = df.dropna(subset=['new_date'])
    df.set_index('new_date', inplace=True)

    # 시간 관련 컬럼 추가
    df['new_year'] = df.index.year
    df['new_quarter'] = df.index.quarter
    df['new_month'] = df.index.month

    # 숫자형 컬럼 변환
    numeric_cols = ['balPayments', 'expDlr', 'expWgt', 'impDlr', 'impWgt']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
        else:
            df[col] = 0.0

    # 정리된 컬럼 선택
    keep_cols = ['hsCode', 'new_year', 'new_quarter', 'new_month', 'statKor',
                'balPayments', 'expDlr', 'expWgt', 'impDlr', 'impWgt', 'root_hs_code']
    clean_df = df[keep_cols].copy()
    clean_df['region'] = region_name

    # 집계 연산
    agg_dict = {
        'balPayments': 'sum',
        'expDlr': 'sum',
        'impDlr': 'sum'
    }

    # 월별 집계
    export_df_by_m = (clean_df.groupby(['root_hs_code', 'new_year', 'new_quarter', 'new_month'])
                      .agg(agg_dict)
                      .reset_index())
    export_df_by_m['region'] = region_name

    # 분기별 집계
    export_df_by_q = (clean_df.groupby(['root_hs_code', 'new_year', 'new_quarter'])
                      .agg(agg_dict)
                      .reset_index())
    export_df_by_q['region'] = region_name

    return export_df_by_q, export_df_by_m

def add_yoy_growth(df, steps):
    """전년동기대비 증가율 계산"""
    df = df.copy()

    if steps == 12:
        # 월 기준
        df['date'] = pd.to_datetime(df['new_year'].astype(str) + '-' +
                                   df['new_month'].astype(str) + '-01') + MonthEnd(0)
    elif steps == 4:
        # 분기 기준
        end_month_map = {1: '03', 2: '06', 3: '09', 4: '12'}
        end_month = df['new_quarter'].map(end_month_map)
        df['date'] = pd.to_datetime(df['new_year'].astype(str) + '-' + end_month + '-01') + MonthEnd(0)
    else:
        raise ValueError("steps는 12(월) 또는 4(분기)여야 합니다.")

    # 정렬 및 YoY 계산
    df = df.sort_values(['root_hs_code', 'date'])

    grouped = df.groupby('root_hs_code')
    df['expDlr_yoy'] = grouped['expDlr'].transform(lambda x: x.pct_change(periods=steps))
    df['impDlr_yoy'] = grouped['impDlr'].transform(lambda x: x.pct_change(periods=steps))

    return df

def reshape_to_long(df):
    """Long format 변환"""
    id_vars = ['date', 'root_hs_code']
    value_vars = ['balPayments', 'expDlr', 'impDlr', 'expDlr_yoy', 'impDlr_yoy']

    # 존재하는 컬럼만 선택
    available_value_vars = [col for col in value_vars if col in df.columns]

    long_df = df.melt(id_vars=id_vars, value_vars=available_value_vars,
                     var_name='indicator', value_name='value')

    # 결측치 제거
    long_df = long_df.dropna(subset=['value'])

    return long_df

def upload_to_db_optimized(df_long, db_info, table_name='korea_monthly_trade_data'):
    """DB 업로드 최적화 - SQLAlchemy 2.0 호환"""
    # from sqlalchemy import create_engine, text
    # import pandas as pd
    # import numpy as np
    # from tqdm import tqdm
    # import logging

    logger = logging.getLogger(__name__)
    CHUNK_SIZE = 1000  # 청크 사이즈 정의

    # 데이터 전처리
    df_long = df_long.copy()
    df_long['date'] = pd.to_datetime(df_long['date'])
    df_long = df_long.replace([np.inf, -np.inf], np.nan)
    df_long = df_long.where(pd.notnull(df_long), None)

    # SQLAlchemy 연결
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}",
        pool_size=20,
        max_overflow=0
    )

    # 테이블 생성 - text()로 감싸기
    create_table_sql = text(f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        `date` DATE,
        `root_hs_code` VARCHAR(20),
        `indicator` VARCHAR(50),
        `value` DOUBLE,
        PRIMARY KEY (`date`, `root_hs_code`, `indicator`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
    """)

    with engine.connect() as conn:
        conn.execute(create_table_sql)
        conn.commit()  # SQLAlchemy 2.0에서는 명시적 commit 필요

    # 중복 확인 - text()로 감싸기
    existing_keys = set()
    try:
        existing_query = text(f"SELECT CONCAT(`date`, '|', `root_hs_code`, '|', `indicator`) as key_combo FROM {table_name}")
        with engine.connect() as conn:
            result = conn.execute(existing_query)
            existing_keys = {row[0] for row in result}
    except Exception as e:
        logger.warning(f"기존 데이터 조회 실패: {e}")

    # 새로운 데이터만 필터링
    df_long['key_combo'] = (df_long['date'].dt.strftime('%Y-%m-%d') + '|' +
                           df_long['root_hs_code'].astype(str) + '|' +
                           df_long['indicator'].astype(str))

    df_to_upload = df_long[~df_long['key_combo'].isin(existing_keys)].drop(columns=['key_combo'])

    # 배치 업로드
    if not df_to_upload.empty:
        logger.info(f"업로드 대상: {len(df_to_upload):,}건")

        for i in tqdm(range(0, len(df_to_upload), CHUNK_SIZE), desc="DB 업로드"):
            chunk = df_to_upload.iloc[i:i + CHUNK_SIZE]
            try:
                chunk.to_sql(name=table_name, con=engine, if_exists='append',
                           index=False, method='multi')
            except Exception as e:
                logger.error(f"청크 {i//CHUNK_SIZE + 1} 업로드 실패: {e}")

        logger.info(f"✅ 총 {len(df_to_upload):,}건 업로드 완료")
    else:
        logger.info("⚠️ 업로드할 새로운 데이터가 없습니다.")

# =============================================================================
# 메인 실행 부분
# =============================================================================

print("🚀 무역 데이터 수집 시작...")
start_time = time.time()

# 기간 리스트 생성
period_list = get_period_list(START_YEAR, END_YEAR, (END_YEAR - START_YEAR + 1) * 12)
print(f"📅 수집 기간: {len(period_list)}개월 ({period_list[0]} ~ {period_list[-1]})")

# 연도별 기간 분할 (API 제한: 1년 12개월)
start_list = [period_list[i] for i in range(0, len(period_list), 12)]
end_list = [period_list[min(i + 11, len(period_list) - 1)] for i in range(0, len(period_list), 12)]

print(f"📊 수집할 HS 코드 수: {len(HS_CODES)}개")
print(f"⚡ 동시 처리 워커: {MAX_WORKERS}개")

# 세션 생성 (연결 재사용)
session = requests.Session()

# 병렬 처리로 데이터 수집
all_dataframes = []
error_list = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # 모든 작업 제출
    future_to_info = {}

    for hs_code in HS_CODES:
        for start, end in zip(start_list, end_list):
            future = executor.submit(
                get_country_export_by_item_optimized, session, SERVICE_KEY, start, end, hs_code
            )
            future_to_info[future] = (hs_code, start, end)

    # 결과 수집
    print("📥 데이터 수집 중...")
    for future in tqdm(as_completed(future_to_info), total=len(future_to_info), desc="API 요청"):
        hs_code, start, end = future_to_info[future]
        try:
            df = future.result(timeout=60)
            if not df.empty:
                all_dataframes.append(df)
        except Exception as e:
            logger.error(f"Error processing {hs_code} ({start}-{end}): {e}")
            error_list.append(f"{hs_code}_{start}_{end}")

# 세션 종료
session.close()

collection_time = time.time() - start_time
print(f"⏱️ 데이터 수집 완료: {collection_time:.2f}초")

if all_dataframes:
    print("🔄 데이터 처리 중...")

    # 전체 데이터 병합
    final_df = pd.concat(all_dataframes, ignore_index=True)
    print(f"📋 총 수집된 레코드: {len(final_df):,}건")

    # 데이터 전처리 및 집계
    processed_q, processed_m = process_and_aggregate(final_df, REGION_NAME)

    print(f"📊 월별 집계 데이터: {len(processed_m):,}건")
    print(f"📊 분기별 집계 데이터: {len(processed_q):,}건")

    # YoY 증가율 계산
    if not processed_m.empty:
        print("📈 전년동기대비 증가율 계산 중...")
        monthly_with_yoy = add_yoy_growth(processed_m, steps=12)
        quarterly_with_yoy = add_yoy_growth(processed_q, steps=4)

        # Long format 변환
        monthly_long = reshape_to_long(monthly_with_yoy)
        quarterly_long = reshape_to_long(quarterly_with_yoy)

        print(f"📊 Long format 월별 데이터: {len(monthly_long):,}건")
        print(f"📊 Long format 분기별 데이터: {len(quarterly_long):,}건")

        # DB 업로드 (선택사항)
        upload_to_db = input("🗃️ 데이터베이스에 업로드하시겠습니까? (y/n): ").lower() == 'y'

        if upload_to_db and DB_INFO['user'] != 'username':  # DB 정보가 실제로 설정된 경우
            print("🗃️ 데이터베이스 업로드 중...")
            upload_to_db_optimized(monthly_long, DB_INFO, 'korea_monthly_trade_data')
            upload_to_db_optimized(quarterly_long, DB_INFO, 'korea_quarterly_trade_data')

        # 결과 출력
        print("\n✅ 처리 완료!")
        print(f"⏱️ 총 소요 시간: {time.time() - start_time:.2f}초")
        print(f"❌ 오류 발생 건수: {len(error_list)}건")

        if error_list:
            print(f"❌ 오류 목록: {error_list[:5]}..." if len(error_list) > 5 else f"❌ 오류 목록: {error_list}")

        # 샘플 데이터 출력
        print(f"\n📋 월별 데이터 샘플:")
        print(monthly_with_yoy.head())

        print(f"\n📋 분기별 데이터 샘플:")
        print(quarterly_with_yoy.head())

        # 전역 변수로 결과 저장 (추후 사용 가능)
        monthly_trade_data = monthly_with_yoy
        quarterly_trade_data = quarterly_with_yoy
        monthly_long_data = monthly_long
        quarterly_long_data = quarterly_long

    else:
        print("⚠️ 처리할 데이터가 없습니다.")

else:
    print("❌ 수집된 데이터가 없습니다.")
    print(f"오류 발생 건수: {len(error_list)}건")

print("\n🎉 스크립트 실행 완료!")

# =============================================================================
# 추가 분석 함수들 (필요시 사용)
# =============================================================================

def plot_column_by_hscode(df, hs_code, col_name, start_date=None, end_date=None):
    """특정 HS 코드의 데이터 시각화"""
    import matplotlib.pyplot as plt

    if 'date' not in df.columns:
        print("❌ 'date' 컬럼이 없습니다. add_yoy_growth()를 먼저 실행하세요.")
        return

    target_df = df[df['root_hs_code'] == hs_code].sort_values('date')

    if target_df.empty:
        print(f"⚠️ root_hs_code {hs_code}에 해당하는 데이터가 없습니다.")
        return

    if col_name not in target_df.columns:
        print(f"❌ '{col_name}' 컬럼이 DataFrame에 없습니다.")
        return

    # 날짜 범위 필터링
    if start_date:
        target_df = target_df[target_df['date'] >= pd.to_datetime(start_date)]
    if end_date:
        target_df = target_df[target_df['date'] <= pd.to_datetime(end_date)]

    if target_df.empty:
        print(f"⚠️ 지정한 날짜 범위에 데이터가 없습니다.")
        return

    # Plot
    plt.figure(figsize=(12, 6))
    plt.plot(target_df['date'], target_df[col_name], marker='o', label=col_name)

    # 마지막 값에 텍스트 표시
    last_x = target_df['date'].iloc[-1]
    last_y = target_df[col_name].iloc[-1]

    if 'yoy' in col_name:
        plt.text(last_x, last_y, f"{last_y * 100:,.2f}%", fontsize=12, ha='left', va='bottom', color='red')
    else:
        plt.text(last_x, last_y, f"{last_y:,.0f}", fontsize=12, ha='left', va='bottom', color='red')

    plt.title(f"{col_name} 추이 (root_hs_code: {hs_code})")
    plt.xlabel("Date")
    plt.ylabel(col_name)
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

print("\n💡 사용 가능한 변수들:")
print("- monthly_trade_data: 월별 무역 데이터")
print("- quarterly_trade_data: 분기별 무역 데이터")
print("- monthly_long_data: Long format 월별 데이터")
print("- quarterly_long_data: Long format 분기별 데이터")
print("\n💡 시각화 예시:")
print("plot_column_by_hscode(monthly_trade_data, '123456', 'expDlr_yoy', '2022-01-01', '2024-12-31')")

🚀 무역 데이터 수집 시작...
📅 수집 기간: 216개월 (200801 ~ 202512)
📊 수집할 HS 코드 수: 499개
⚡ 동시 처리 워커: 15개
📥 데이터 수집 중...


API 요청:   0%|          | 2/8982 [00:01<1:10:09,  2.13it/s]2025-10-26 23:45:50,608 - ERROR - Unexpected error for HS 2208904000: name 'json' is not defined
2025-10-26 23:45:50,775 - ERROR - Unexpected error for HS 2208904000: name 'json' is not defined
API 요청:   0%|          | 4/8982 [00:01<37:50,  3.95it/s]  2025-10-26 23:45:50,851 - ERROR - Unexpected error for HS 2208904000: name 'json' is not defined
2025-10-26 23:45:50,866 - ERROR - Unexpected error for HS 2208904000: name 'json' is not defined
2025-10-26 23:45:50,887 - ERROR - Unexpected error for HS 2208904000: name 'json' is not defined
API 요청:   0%|          | 7/8982 [00:01<19:20,  7.74it/s]2025-10-26 23:45:50,988 - ERROR - Unexpected error for HS 2208904000: name 'json' is not defined
2025-10-26 23:45:51,251 - ERROR - Unexpected error for HS 2208904000: name 'json' is not defined
API 요청:   0%|          | 9/8982 [00:01<22:07,  6.76it/s]2025-10-26 23:45:51,367 - ERROR - Unexpected error for HS 2208904000: name 'json' is not defi